# XGBoost on Descriptors (Representation A)


## Step 1 - Prepare the data for XGBoost

In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [13]:
features_df = pd.read_csv("training_features.csv")
features_df.head()

,Unnamed: 0,INDEX,SMILES,ACTIVE,MolFromSmiles,NumAtoms,NumHeavyAtoms,NumBonds,fr_Al_COO,fr_Al_OH,...,CalcNumRings,CalcNumRotatableBonds,CalcNumSaturatedCarbocycles,CalcNumSaturatedHeterocycles,CalcNumSaturatedRings,CalcNumSpiroAtoms,CalcNumUnspecifiedAtomStereoCenters,CalcPhi,CalcTPSA,_CalcMolWt
0,0,1,O=C(Nc1ccc2c(c1)OCCO2)C1CCN(c2ncccn2)CC1,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0040>,25,25,28,0,0,...,4,3,0,1,1,0,0,4.368063,76.58,340.383
1,1,2,COCCCN1C(=O)C2C(C(=O)Nc3cccc(Cl)c3)C3C=CC2(O3)...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea00b0>,35,35,39,0,0,...,5,8,1,2,3,1,5,6.877647,96.97,502.011
2,2,3,CCSc1ncc(Cl)c(C(=O)Nc2ccccc2C)n1,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0120>,20,20,21,0,0,...,2,4,0,0,0,0,0,4.977937,54.88,307.806
3,3,4,COc1ccc2cc(/C=N/NC(=O)CN(c3ccccc3C)S(=O)(=O)c3...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0190>,36,36,39,0,0,...,4,8,0,0,0,0,0,7.516779,100.96,523.014
4,4,5,CCCC(=O)Nc1nc2ccc(NC(=O)c3c(F)c(F)c(OC)c(F)c3F...,0.0,<rdkit.Chem.rdchem.Mol object at 0x116ea0200>,30,30,32,0,0,...,3,6,0,0,0,0,0,6.202841,80.32,441.406


## Step 2 - Split into X (features) and y (targets)

In [14]:
features_df.columns

Index(['Unnamed: 0', 'INDEX', 'SMILES', 'ACTIVE', 'MolFromSmiles', 'NumAtoms',
       'NumHeavyAtoms', 'NumBonds', 'fr_Al_COO', 'fr_Al_OH',
       ...
       'CalcNumRings', 'CalcNumRotatableBonds', 'CalcNumSaturatedCarbocycles',
       'CalcNumSaturatedHeterocycles', 'CalcNumSaturatedRings',
       'CalcNumSpiroAtoms', 'CalcNumUnspecifiedAtomStereoCenters', 'CalcPhi',
       'CalcTPSA', '_CalcMolWt'],
      dtype='object', length=157)

In [15]:
TARGET_COL = "ACTIVE"  
y = features_df[TARGET_COL]

In [16]:
non_feature_cols = [TARGET_COL, "SMILES", "MolFromSmiles", "Unnamed: 0", "INDEX"]
non_feature_cols = [c for c in non_feature_cols if c in features_df.columns]

y = features_df[TARGET_COL]
X = features_df.drop(columns=non_feature_cols)
X = X.select_dtypes(include=[np.number])

X.shape, y.shape

((202895, 152), (202895,))

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((162316, 152), (40579, 152), (162316,), (40579,))

## Step 3 - Handle missing values

In [18]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_imputed.shape, X_test_imputed.shape

((162316, 152), (40579, 152))

In [19]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np

def get_repeated_cv_auc(estimator, X, y, n_splits=5, n_repeats=3, random_state=42):
    rskf = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=random_state
    )
    scores = cross_val_score(
        estimator,
        X,
        y,
        scoring="roc_auc",
        cv=rskf,
        n_jobs=1
    )
    return scores

## Step 4 - Train

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_scores = cross_val_score(
    xgb_clf,
    X_train_imputed,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC per fold:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


AUC per fold: [0.88680029 0.88589562 0.88270769 0.89442527 0.89494447]
Mean AUC: 0.8889546672640097
Std AUC: 0.004875021407420356


In [21]:
xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1
)

auc_scores = get_repeated_cv_auc(xgb_clf, X_train_imputed, y_train, n_splits=5, n_repeats=3)

print("AUC repeated CV:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


AUC repeated CV: [0.88748276 0.88616849 0.88245525 0.89343908 0.89597563 0.88404952
 0.88991899 0.88492019 0.88869735 0.89245572 0.89622187 0.89145887
 0.87962062 0.88607784 0.88852534]
Mean AUC: 0.8884978344603195
Std AUC: 0.004666936209200163


In [22]:
import scipy.stats as st
mean = auc_scores.mean()
std = auc_scores.std(ddof=1)
n = len(auc_scores)
ci_low, ci_high = st.t.interval(0.95, df=n-1, loc=mean, scale=std/np.sqrt(n))

print("95% CI (mean AUC, t-approx):", (ci_low, ci_high))

95% CI (mean AUC, t-approx): (np.float64(0.8858226609689592), np.float64(0.8911730079516798))


In [23]:
def bootstrap_ci(data, n_boot=2000, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    boots = []
    data = np.asarray(data)
    for _ in range(n_boot):
        sample = rng.choice(data, size=len(data), replace=True)
        boots.append(sample.mean())
    boots = np.array(boots)
    low = np.quantile(boots, alpha/2)
    high = np.quantile(boots, 1-alpha/2)
    return low, high

print("95% bootstrap CI (mean AUC):", bootstrap_ci(auc_scores))


95% bootstrap CI (mean AUC): (np.float64(0.8861097700537016), np.float64(0.8909334283715088))


## Step 5 - Tuning

In [24]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_dist = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_imputed, y_train)

print("Best AUC (cv=5):", random_search.best_score_)
print("Best params:", random_search.best_params_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best AUC (cv=5): 0.8946033423706214
Best params: {'subsample': 0.8, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


A mean AUC over k folds plus a standard deviation is not enough to claim that your model is better than alternatives.

In [25]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

best_xgb = random_search.best_estimator_

cv10 = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

auc_scores_10 = cross_val_score(
    best_xgb,
    X_train_imputed,
    y_train,
    cv=cv10,
    scoring="roc_auc",
    n_jobs=-1
)

print("10-fold AUC mean:", auc_scores_10.mean())
print("10-fold AUC std:", auc_scores_10.std())


10-fold AUC mean: 0.8964873207113839
10-fold AUC std: 0.006087122904579564


In [26]:
from sklearn.metrics import roc_auc_score

best_xgb.fit(X_train_imputed, y_train)
y_proba_test = best_xgb.predict_proba(X_test_imputed)[:, 1]
test_auc = roc_auc_score(y_test, y_proba_test)

print("Held-out test AUC (20%):", test_auc)


Held-out test AUC (20%): 0.8992302261763244


In [27]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_imputed, y_train)

y_dummy_proba = dummy.predict_proba(X_test_imputed)[:, 1]
auc_dummy = roc_auc_score(y_test, y_dummy_proba)

print("Baseline DummyClassifier AUC:", auc_dummy)
print("XGBoost best model Test AUC:", test_auc)

Baseline DummyClassifier AUC: 0.5
XGBoost best model Test AUC: 0.8992302261763244


## Results and insights

### Model performance with cross validation

We first evaluated the XGBoost model using 5 fold cross validation on the training set:

- AUC per fold:  
  `[0.8868, 0.8859, 0.8827, 0.8944, 0.8949]`  
- Mean AUC: **0.8890**  
- Standard deviation: **0.0049**

To better assess stability, we repeated cross validation multiple times:

- Repeated CV AUC values:  
  `[0.8875, 0.8862, 0.8825, 0.8934, 0.8960, 0.8840, 0.8899, 0.8849, 0.8887, 0.8925, 0.8962, 0.8915, 0.8796, 0.8861, 0.8885]`
- Mean AUC: **0.8885**  
- Standard deviation: **0.0047**  
- 95% CI for the mean AUC (t approximation): **[0.8858, 0.8912]**  
- 95% bootstrap CI for the mean AUC: **[0.8861, 0.8909]**

The narrow confidence intervals and low standard deviation indicate that the model performance is stable across different splits of the data.

---

### Hyperparameter tuning

We performed hyperparameter tuning with 5 fold cross validation over 20 candidate configurations (100 fits in total). The best configuration achieved:

- Best CV AUC (5 fold): **0.8946**  
- Best hyperparameters:  
  - `learning_rate = 0.05`  
  - `n_estimators = 400`  
  - `max_depth = 8`  
  - `min_child_weight = 5`  
  - `subsample = 0.8`  
  - `colsample_bytree = 0.8`

We then re evaluated the best model with a fresh 10 fold cross validation:

- 10 fold mean AUC: **0.8965**  
- 10 fold AUC standard deviation: **0.0061**

This confirms that tuning provided a consistent performance gain compared to the initial configuration.

---

### Held out test set performance

Finally, we evaluated the tuned model on a held out test set (20 percent of the data):

- Held out test AUC: **0.8992**

As a reference, a `DummyClassifier` baseline that predicts according to the class distribution achieved:

- Baseline AUC: **0.5**

The XGBoost model clearly outperforms the baseline and the test AUC is very close to the cross validation estimates, which suggests:

- Good generalization to unseen data  
- Limited overfitting, since test performance is in line with CV performance  

---

### Summary

- The XGBoost model reaches a mean AUC around **0.89** with very low variance across folds.  
- Hyperparameter tuning further improves AUC to about **0.896** in cross validation.  
- The final held out test AUC of **0.899** confirms that the tuned model generalizes well and provides a strong improvement over a random baseline (AUC 0.5).
